# DAD (Dynamic Affinity Dock) — Colab Environment Setup

This notebook prepares Google Colab for running the DAD pipeline. It installs dependencies, mounts the `dad` package from your local environment, and runs a smoke test to verify correctness.

## Cell 1: Install Core Dependencies

Install pip packages required for DAD. Skip conda (use pip + system packages).

In [ ]:
# Install system packages
!apt-get update -qq && apt-get install -y -qq \
  build-essential \
  git \
  wget \
  curl \
  cmake \
  libopenmpi-dev \
  openmpi-bin \
  libhdf5-serial-dev \
  libhdf5-mpi-dev > /dev/null 2>&1

# Upgrade pip
!pip install --upgrade pip --quiet

# Install core dependencies
!pip install \
  biopython>=1.81 \
  rdkit>=2023.09 \
  openbabel-wheel>=3.1 \
  py3Dmol>=2.0 \
  pandas>=2.0 \
  numpy>=1.24 \
  scipy>=1.11 \
  matplotlib>=3.8 \
  seaborn>=0.13 \
  pyyaml>=6.0 \
  plip>=2.2.2 \
  deeptmhmm>=1.0 \
  posebusters>=0.1.0 \
  dimorphite-dl>=1.3.6 \
  mmcif-pdbx>=2.0 \
  --quiet

print("✓ Core dependencies installed.")

## Cell 2: Mount Google Drive & Import DAD Package

Mount your Google Drive (or local filesystem), then import the `dad` package from the DAD project directory.

In [ ]:
from google.colab import drive
import sys
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

# Path to DAD project (adjust if stored elsewhere)
# Example: /content/drive/MyDrive/DAD/
dad_project_root = Path('/content/drive/MyDrive/DAD')

# Add dad package to sys.path
dad_package_path = dad_project_root / '06_Report' / 'Mr_Pipeline'
if dad_package_path.exists():
    sys.path.insert(0, str(dad_project_root / '06_Report'))
    print(f"✓ DAD package path added: {dad_package_path}")
else:
    print(f"⚠ DAD package path not found: {dad_package_path}")
    print(f"  Ensure DAD repo is in: {dad_project_root}")

# Test import
try:
    import dad.core
    print("✓ dad.core package imported successfully.")
except ImportError as e:
    print(f"⚠ Failed to import dad.core: {e}")
    print(f"  Verify that {dad_package_path / 'dad'} contains __init__.py")

## Cell 3: Smoke Test

Verify that core dependencies and the `dad` package are working correctly.

In [ ]:
import sys
import importlib

# List of core packages to verify
required_packages = [
    ('biopython', 'Bio'),
    ('rdkit', 'rdkit'),
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('py3Dmol', 'py3Dmol'),
    ('plip', 'plip'),
    ('deeptmhmm', 'deeptmhmm'),
    ('posebusters', 'posebusters'),
]

print("Smoke test: Verifying package imports...\n")
failed = []
for package_name, import_name in required_packages:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✓ {package_name:20} ({version})")
    except ImportError as e:
        print(f"✗ {package_name:20} — {e}")
        failed.append(package_name)

print()

# Test dad.core import
try:
    import dad.core
    print(f"✓ dad.core                 (ready)")
except ImportError as e:
    print(f"✗ dad.core                 — {e}")
    failed.append('dad.core')

print()
if failed:
    print(f"⚠ {len(failed)} package(s) failed to import. Check Cell 1 & 2 output above.")
else:
    print("✓ All packages imported successfully. Ready to use DAD pipeline.")